In [ ]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression 
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder.master("local[*]").appName("practice").getOrCreate()

df = spark.read.csv("train.csv", header=True, inferSchema=True)
gender = spark.read.csv("gender_submission.csv", header=True, inferSchema=True)

df.printSchema()
gender.printSchema()

n = df.count()
print("Len", n, gender.count())

for c in df.columns:
    if df.filter(F.col(c).isNull()).count() > n/2:
        df = df.drop(c)

df = df.withColumn(
    "Sex", F.when(F.col("Sex") == "male", 0.0).otherwise(1.0) 
)

df = df.join(gender, "PassengerId", "left")

cols = ["Age", "Sex"]
filled = ["Age_filled", "Sex_filled"]

train, test = df.randomSplit([0.8], [0.2], seed=42)

pipeline = Pipeline(stages=[
    Imputer(strategy="median", inputCols=cols, outputCols=filled),
    VectorAssembler(inputCols=filled, outputCol="vector"),
    StandardScaler(inputCol="vector", outputCol="feature", withMean=True),
    LogisticRegression(labelCol="Survived", maxIter=0)
])

model = pipeline.fit(train)

model.transform(df).write.mode("overwrite").parquet("output")

predictions = model.transform(test)
predictions.select("Survived", "prediction").show()

evaluator = MulticlassClassificationEvaluator(labelCol="Survived")

print(evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"}))


```bash
git init
dvc init
dvc add data/iris.csv
```

In [ ]:
import pandas as pd
import joblib
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

data = pd.read_csv("data/iris.csv")

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns="target"), data["target"],
    test_size=0.2, random_state=42
)

mlflow.set_experiment("iris")

with mlflow.start_run():
    params = {"n_estimators": 100}
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)
    mlflow.sklearn.log_model(model, name="model")

    joblib.dump(model, "model.joblib")

**dvc.yaml**
```yaml
stages:
    train:
        cmd: python train.py
        deps:
            - train.py
            - data/iris.csv
        outs:
            - model.joblib
```

```bash
dvc repro
git add .
git commit -m "track training"
```

# ML CI/CD

```bash
git init
```

In [ ]:
# train.py
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

def train():
    X, y = load_iris(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)

    return model, X_test, y_test

In [ ]:
test.py
from train import train

def test_accuracy():
    model, X_test, y_test = train()
    assert model.score(X_test, y_test) >= 0.8

**`.github/workflows/ci.yml`:**
```yaml
name: ML Tests

on: [push, pull_request]

jobs: 
    test:
        runs-on: ubuntu-latest 

        steps:
            - uses: actions/checkout@v6

            - uses: actions/setup-python@v5
              with:
                python-version: "3.12"

            - run: pip install scikit-learn pytest
            - run: python -m pytest test.py
```

git commands:
```bash
git add .
git commit -m "add model"
git branch -M main
git remote add origin ""
git push -u origin main
```

### implement a MapReduce word count with a mapper and reducer?

In [ ]:
import re
from collections import Counter 

text = "Hello hello world"

mapped = [(word, 1) for word in re.findall(r"\w+", text.lower())]

counts = Counter()

for word, count in mapped:
    counts[word] += count

print(counts)

In [ ]:
from pyspark.sql import SparkSession, functions as F 

spark = SparkSession.builder.master("local[*]").appName("practice").getOrCreate()

data = [
    (101, "South", "Laptop", 2, 45000),
    (102, "North", "Mobile", 5, 15000),
    (103, "South", "Tablet", 3, 20000),
    (104, "East", "Laptop", 1, 50000),
    (105, "West", "Mobile", 4, 12000),
    (106, "North", "Laptop", 2, 55000),
    (107, "East", "Tablet", 5, 10000),
    (108, "West", "Laptop", 1, 60000),
    (109, "South", "Mobile", 6, 10000),
    (110, "North", "Tablet", 4, 18000)
]

columns = ["order_id", "region", "product", "quantity", "unit_price"]

df = spark.createDataFrame(data, columns)

df.show()

In [ ]:
df = df.withColumn("total_amount", df.quantity * df.unit_price)
df = df.filter(df.total_amount > 5000)

result = df.groupBy("region").agg(
    F.sum("total_amount").alias("total_sales"),
    F.count("*").alias("order_count")
)

result.orderBy(F.desc("total_sales")).show(2)

In [ ]:
df.createOrReplaceTempView("passengers")

spark.sql("""""")

logs= ["err crassh"]

rdd = spark.sparkContext.parallelize(logs)

print(rdd.filter(lambda x: x.startswith("err")).collect())

counts = (
    rdd.map(lambda x: (x.split()[0], 1)).reduceByKey(lambda a, b: a + b)
)

print(counts.collect())

spark.stop()

In [ ]:
from tfx import v1 as tfx
from tfx.orchestration.experimental.iteractive.interactive_context


In [3]:
import tarfile 
import sagemaker 
from sagemaker.sklearn.model import SKLearnModel

session = sagemaker.Session()

with tarfile.open("model.tar.gz", "w:gz") as archive:
    archive.add("model.joblib")

model = SKLearnModel(
    model_data=session.upload("model.tar.gz"),
    role="ARN",
    entry_point="inference.py",
)

predictor = model.deploy(
    initial_instance_count=1,
    isinstance_type="ml.m5.large",
)

SyntaxError: expected argument value expression (2997337674.py, line 13)